# Amazon Bedrock: Foundation Models, Model Lifecycle, and Inference Parameters

## 1. Theory & Real-World Context

**Foundation Models (FMs)** are large-scale machine learning models trained on vast, unlabelled datasets. Unlike traditional ML models built for narrow tasks (e.g., predicting house prices), FMs learn general data representations, allowing them to be fine-tuned or prompted for a wide array of downstream tasks (text generation, coding, image creation).

Amazon Bedrock provides a unified, serverless API to access industry-leading FMs (from Anthropic, Meta, Mistral, Cohere, and Amazon) along with an **Amazon Bedrock Marketplace** for specialized, domain-specific models (e.g., biomedical or financial FMs).

### The Bedrock Model Lifecycle

Understanding the lifecycle is critical for production AI pipelines:

1. **Active:** The recommended, fully supported version.
2. **Legacy:** An older version being phased out. Existing apps continue to work (for at least 6 months), but no new Provisioned Throughput can be allocated. A **Public Extended Access Period** may apply, introducing higher inference costs to incentivize migration.
3. **End-of-Life (EOL):** The model is fully retired. Requests will fail. **Migration is manual**—you must update your code to point to a new model ID.

### Inference Parameters (The "Missing Tabs")

Inference parameters are the levers you pull to shape the output of a foundation model.

**Randomness & Diversity:**

* **Temperature (0.0 to 1.0+):** Modulates the probability distribution of the next token. Low values (~0.0) make the model highly deterministic and factual. High values (~0.9) flatten the probabilities, allowing the model to pick less common words, generating more creative or varied text.
* **Top K:** An absolute cutoff. The model restricts its selection pool to only the *K* most probable next tokens, discarding the long tail of unlikely words.
* **Top P (Nucleus Sampling):** A dynamic cutoff. The model restricts its selection pool to the smallest set of tokens whose cumulative probabilities equal *P*. For example, `Top P = 0.9` means the model considers only the tokens making up the top 90% of the probability mass.

**Length & Boundaries:**

* **Response Length (Max Tokens):** A hard limit on the maximum number of tokens the model can generate in a single response.
* **Length Penalty:** A modifier that discourages the model from generating excessively long outputs without hitting a hard, abrupt cutoff.
* **Stop Sequences:** Specific string patterns (e.g., `\n\nUser:`, `###`) that force the model to immediately halt generation when encountered, preventing run-on text or hallucinated conversations.

> **Real-World AI Engineering Context:** For strict data extraction workflows (e.g., pulling JSON entities from PDFs), you set `Temperature = 0.0` and `Top P = 0.0` to force maximum determinism. For generating synthetic training data or creative copywriting, you increase `Temperature = 0.7 - 0.9` to introduce variance.

---

## 2. Visual Architecture

```mermaid
flowchart TD
    A[Model Calculates Probabilities<br/>for all possible next tokens] --> B{Top K Filter}
    B -- Keeps the Top 'K' tokens by rank --> C{Top P Filter}
    C -- Keeps tokens until cumulative<br/>probability equals 'P' --> D{Temperature Scaling}
    D -- Reshapes probability curve<br/>(Sharpens or Flattens) --> E[Sample Next Token]
    
    E --> F{Stop Sequence<br/>Generated?}
    F -- Yes --> G([End Generation])
    F -- No --> H{Max Token Limit<br/>Reached?}
    
    H -- Yes --> G
    H -- No --> A

```

---

## 3. Console Hands-On

**Objective:** Observe the impact of inference parameters directly in the console.

1. Navigate to the **Amazon Bedrock Console** $\rightarrow$ **Playgrounds** $\rightarrow$ **Chat**.
2. Click **Select model** and choose **Anthropic $\rightarrow$ Claude 3 Haiku**.
3. In the right-hand **Configurations** panel, set **Temperature** to `0.0`, **Top P** to `0.001`, and **Maximum length** to `500`.
4. Enter the prompt: `Write a one-paragraph story about an AI discovering the internet.` Note the straightforward, dry response.
5. Change **Temperature** to `1.0` and **Top P** to `0.999`. Rerun the prompt. Notice the drastic shift in vocabulary, tone, and creativity.
6. Under **Stop sequences**, type the word `data` and press Enter. Rerun the prompt. The model will abruptly stop generating the exact moment it tries to output the word "data".

---

## 5. Practice Challenges

### Challenge 1: The Hallucination Breaker

**Scenario:** You are building an AI data pipeline that converts raw support emails into strict SQL queries using Amazon Bedrock. Currently, the model is generating valid SQL, but occasionally appends conversational fluff like *"Here is your query:"* or continues generating a second, unrequested query.

* **Task:** Identify the exact three inference parameters you should adjust (and their values) to force strict, code-only determinism and halt generation after the first query finishes.

### Challenge 2: Programmatic Lifecycle Monitoring

**Scenario:** Your company depends heavily on `anthropic.claude-v2` for a production pipeline.

* **Task:** Using the `boto3.client('bedrock')` control plane client, write a function that calls `list_foundation_models()`. Filter the response to find your specific model ID and programmatically print an alert if the `modelLifecycle.status` transitions from `ACTIVE` to `LEGACY`.

---

## 6. Industry Standards & Tips

* **The "One Lever" Rule:** Never tune Temperature, Top P, and Top K simultaneously. Industry standard is to leave Top K and Top P at their defaults and only adjust **Temperature**. If you require even finer control, adjust Top P, but leave Top K alone.
* **Lifecycle CI/CD Checks:** Never let an EOL date catch you off guard. Write an automated integration test that queries the `GetFoundationModel` API for your active models. If a model transitions to `Legacy`, fail the build gracefully with a warning to the MLOps team that a 6-month migration clock has started.
* **Prompt vs. Parameters:** If your model is consistently outputting the wrong format, inference parameters cannot fix it. Parameters control *how* the model selects words, not *what* the model understands. Fix formatting issues with Prompt Engineering (e.g., Few-Shot examples); use inference parameters strictly for controlling variance.

---

## 7. Interview Q&A (Targeted for AI Engineers)

### Q1: How do Top K and Top P differ mathematically when shaping a model's response?

> **Answer:** Top K is a static filter that restricts the model's vocabulary to a fixed number of tokens (e.g., the top 50), regardless of how much probability mass they actually hold. Top P (Nucleus Sampling) is dynamic; it restricts the vocabulary to the smallest set of tokens whose cumulative probability reaches the threshold $P$. If the model is highly confident in one word (e.g., 95% probability), a Top P of 0.90 will only consider that single word, ignoring the rest of the pool.

### Q2: We are heavily relying on an Amazon Bedrock foundation model that just transitioned to the `Legacy` state. What are the immediate impacts on our infrastructure, and what is our timeline?

> **Answer:** Existing API calls will not break immediately; the model remains available for at least 6 months. However, we cannot purchase any new Provisioned Throughput for it, and no new AWS accounts can adopt it. If it enters the Public Extended Access Period, we may incur higher inference costs. We must prioritize migrating our code to a newer, `Active` model before the End-of-Life (EOL) date, after which all API requests will return hard failures.

### Q3: You want to prevent a model from generating overly long, verbose responses. Should you rely entirely on the `Max Tokens` parameter?

> **Answer:** No. While `Max Tokens` provides a hard safety limit to prevent runaway API costs, relying on it alone will result in the model abruptly stopping mid-sentence when the limit is hit. Instead, you should combine a reasonable `Max Tokens` limit with a higher **Length Penalty** (to naturally encourage the model to output shorter sequences) and configure proper **Stop Sequences** so the model knows exactly when its generation task is complete.

---

## Boto3 Implementation (optional)

Below is a production-ready script using the modern Bedrock `Converse` API, which standardizes inference parameters (Temperature, Top P, Max Tokens, Stop Sequences) across all foundation models.

```python
import logging
import boto3
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

class BedrockInferenceManager:
    """Manages AI model invocations using standardized Bedrock inference parameters."""
    
    def __init__(self, region_name: str = "us-east-1"):
        self.bedrock_runtime = boto3.client("bedrock-runtime", region_name=region_name)

    def generate_deterministic_response(self, prompt: str, model_id: str) -> str:
        """
        Invokes a Bedrock model configured for deterministic, factual output 
        (e.g., for RAG pipelines or data extraction).
        """
        # The Converse API standardizes parameters across all model providers
        inference_config = {
            "temperature": 0.0,      # Maximum strictness; no creativity
            "topP": 0.1,             # Restrict to the highest probability tokens
            "maxTokens": 512,        # Cap the output length
            "stopSequences": ["\n\nUser:", "```\n\n"] # Stop if it hallucinates boundaries
        }
        
        messages = [{"role": "user", "content": [{"text": prompt}]}]

        try:
            logger.info(f"Invoking {model_id} with deterministic settings...")
            response = self.bedrock_runtime.converse(
                modelId=model_id,
                messages=messages,
                inferenceConfig=inference_config
            )
            
            output_text = response['output']['message']['content'][0]['text']
            stop_reason = response['stopReason']
            
            logger.info(f"Generation stopped due to: {stop_reason}")
            return output_text
            
        except ClientError as e:
            logger.error(f"Bedrock invocation failed: {e.response['Error']['Message']}")
            raise

if __name__ == "__main__":
    MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"
    manager = BedrockInferenceManager()
    
    result = manager.generate_deterministic_response(
        prompt="Extract the key entities from the following text and format as JSON: [Text body]",
        model_id=MODEL_ID
    )
    print(result)

```